# 01 — LandCover.ai Data Exploration

This notebook downloads the **LandCover.ai** dataset (~1.4 GB), explores real high-resolution orthophotos and segmentation masks, computes vegetation indices, and previews the tiled chips used for training.

**Run on Google Colab** for free GPU access and fast downloads.

In [ ]:
# Install dependencies
!pip install -q rasterio numpy matplotlib opencv-python-headless Pillow

In [ ]:
import os
import zipfile
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from collections import Counter
from pathlib import Path

## 1. Download LandCover.ai

In [ ]:
DATA_DIR = Path('landcoverai')
DATA_DIR.mkdir(exist_ok=True)

ZIP_URL = 'https://landcover.ai.linuxpolska.com/download/landcover.ai.v1.zip'
ZIP_PATH = DATA_DIR / 'landcover.ai.v1.zip'

if not (DATA_DIR / 'images').exists():
    print('Downloading LandCover.ai (~1.4 GB)...')
    !wget -q --show-progress -O {ZIP_PATH} {ZIP_URL}
    print('Extracting...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(DATA_DIR)
    ZIP_PATH.unlink()  # remove zip to save space
    print('Done!')
else:
    print('Dataset already downloaded.')

IMAGE_DIR = DATA_DIR / 'images'
MASK_DIR = DATA_DIR / 'masks'

image_files = sorted(IMAGE_DIR.glob('*.tif'))
mask_files = sorted(MASK_DIR.glob('*.tif'))
print(f'Found {len(image_files)} images, {len(mask_files)} masks')

## 2. Visualize RGB Orthophotos and Masks

LandCover.ai classes:
- **0** — Background
- **1** — Building
- **2** — Woodland
- **3** — Water
- **4** — Road

In [ ]:
CLASS_NAMES = {0: 'Background', 1: 'Building', 2: 'Woodland', 3: 'Water', 4: 'Road'}
CLASS_COLORS = {
    0: [0, 0, 0],        # black
    1: [255, 0, 0],      # red
    2: [0, 180, 0],      # green
    3: [0, 0, 255],      # blue
    4: [255, 255, 0],    # yellow
}
NUM_CLASSES = 5

def colorize_mask(mask):
    """Convert class-index mask to RGB."""
    h, w = mask.shape
    rgb = np.zeros((h, w, 3), dtype=np.uint8)
    for cls_id, color in CLASS_COLORS.items():
        rgb[mask == cls_id] = color
    return rgb

# Show 4 random samples
rng = np.random.default_rng(42)
indices = rng.choice(len(image_files), size=4, replace=False)

fig, axes = plt.subplots(4, 2, figsize=(14, 24))
for row, idx in enumerate(indices):
    img = np.array(Image.open(image_files[idx]))
    msk = np.array(Image.open(mask_files[idx]))

    axes[row, 0].imshow(img)
    axes[row, 0].set_title(f'Image: {image_files[idx].name}  ({img.shape[1]}x{img.shape[0]})')
    axes[row, 0].axis('off')

    axes[row, 1].imshow(colorize_mask(msk))
    axes[row, 1].set_title('Mask')
    axes[row, 1].axis('off')

plt.tight_layout()
plt.show()

## 3. Class Distribution

In [ ]:
# Compute class distribution across all masks
pixel_counts = Counter()

for mf in mask_files:
    msk = np.array(Image.open(mf))
    unique, counts = np.unique(msk, return_counts=True)
    for u, c in zip(unique, counts):
        if u < NUM_CLASSES:
            pixel_counts[int(u)] += int(c)

total = sum(pixel_counts.values())
print('Class distribution (all masks):')
for cls_id in range(NUM_CLASSES):
    pct = pixel_counts[cls_id] / total * 100
    print(f'  {cls_id} {CLASS_NAMES[cls_id]:>12s}: {pct:6.2f}%  ({pixel_counts[cls_id]:,} px)')

# Bar chart
labels = [CLASS_NAMES[i] for i in range(NUM_CLASSES)]
values = [pixel_counts[i] / total * 100 for i in range(NUM_CLASSES)]
colors = [np.array(CLASS_COLORS[i]) / 255.0 for i in range(NUM_CLASSES)]
colors[0] = [0.3, 0.3, 0.3]  # make background visible on white bg

plt.figure(figsize=(8, 4))
plt.bar(labels, values, color=colors, edgecolor='black')
plt.ylabel('Percentage of pixels')
plt.title('LandCover.ai — Class Distribution')
plt.tight_layout()
plt.show()

## 4. NDVI on Real Imagery

LandCover.ai provides RGB images (no NIR band), so we approximate a pseudo-NDVI using the green and red channels: `pseudo_NDVI = (G - R) / (G + R + eps)`.
This highlights vegetated areas that appear green in the orthophotos.

In [ ]:
# Pick a sample image with likely vegetation
sample_img = np.array(Image.open(image_files[0])).astype(np.float32)
red = sample_img[:, :, 0]
green = sample_img[:, :, 1]

eps = 1e-6
pseudo_ndvi = (green - red) / (green + red + eps)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
axes[0].imshow(sample_img.astype(np.uint8))
axes[0].set_title('RGB')
axes[0].axis('off')

im = axes[1].imshow(pseudo_ndvi, cmap='RdYlGn', vmin=-0.5, vmax=0.5)
axes[1].set_title('Pseudo-NDVI (G−R)/(G+R)')
axes[1].axis('off')
plt.colorbar(im, ax=axes[1], fraction=0.046)

msk = np.array(Image.open(mask_files[0]))
axes[2].imshow(colorize_mask(msk))
axes[2].set_title('Ground Truth Mask')
axes[2].axis('off')

plt.tight_layout()
plt.show()

print(f'Pseudo-NDVI range: [{pseudo_ndvi.min():.3f}, {pseudo_ndvi.max():.3f}]')
print(f'Mean: {pseudo_ndvi.mean():.3f}')
print(f'"Green" coverage (pseudo-NDVI > 0.1): {(pseudo_ndvi > 0.1).mean() * 100:.1f}%')

## 5. Preview Tiles (512×512 → 256×256)

LandCover.ai ships with a `split.py` script that chips the large orthophotos into 512×512 tiles. For U-Net training we resize them to 256×256. Here we preview what those tiles look like.

In [ ]:
TILE_SIZE = 512
RESIZE_TO = 256

# Tile a sample image
sample_img = np.array(Image.open(image_files[0]))
sample_msk = np.array(Image.open(mask_files[0]))
h, w = sample_img.shape[:2]

tiles_img, tiles_msk = [], []
for y in range(0, h - TILE_SIZE + 1, TILE_SIZE):
    for x in range(0, w - TILE_SIZE + 1, TILE_SIZE):
        tile_img = sample_img[y:y+TILE_SIZE, x:x+TILE_SIZE]
        tile_msk = sample_msk[y:y+TILE_SIZE, x:x+TILE_SIZE]
        tiles_img.append(tile_img)
        tiles_msk.append(tile_msk)

print(f'Original image: {w}x{h} → {len(tiles_img)} tiles of {TILE_SIZE}x{TILE_SIZE}')

# Show first 8 tiles resized to 256x256
n_show = min(8, len(tiles_img))
fig, axes = plt.subplots(2, n_show, figsize=(3 * n_show, 6))
for i in range(n_show):
    resized_img = np.array(Image.fromarray(tiles_img[i]).resize((RESIZE_TO, RESIZE_TO), Image.BILINEAR))
    resized_msk = np.array(Image.fromarray(tiles_msk[i]).resize((RESIZE_TO, RESIZE_TO), Image.NEAREST))

    axes[0, i].imshow(resized_img)
    axes[0, i].set_title(f'Tile {i}', fontsize=9)
    axes[0, i].axis('off')

    axes[1, i].imshow(colorize_mask(resized_msk))
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Image', fontsize=11)
axes[1, 0].set_ylabel('Mask', fontsize=11)
plt.suptitle(f'Sample tiles resized to {RESIZE_TO}×{RESIZE_TO}', fontsize=13)
plt.tight_layout()
plt.show()

## Summary

- LandCover.ai contains high-resolution RGB orthophotos with 5-class pixel-level masks.
- The dataset is heavily imbalanced (background dominates), so training should use Dice + CrossEntropy loss.
- Images are chipped to 512×512 (via the bundled `split.py`) and resized to 256×256 for U-Net training.
- Proceed to **`02_unet_training.ipynb`** to train the segmentation model on this data.